#Libraries

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

#PHASE 7 - UNDERSAMPLING & MODEL COMPARISON

In [2]:
#Random Undersampling was applied to balance the classes by
#reducing samples from the majority class.

#The objective is to evaluate whether undersampling improves
#recall compared to SMOTE.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
base_path = "/content/drive/MyDrive/Customer-Satisfaction-Retail-Analytics/data/Processed Data/"

X_train = pd.read_csv(base_path + "X_train_scaled.csv")
X_test = pd.read_csv(base_path + "X_test_scaled.csv")

y_train = pd.read_csv(base_path + "y_train.csv").squeeze()
y_test = pd.read_csv(base_path + "y_test.csv").squeeze()

In [5]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(75564, 20)
(18891, 20)
(75564,)
(18891,)


##Undersampling

In [6]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(
    random_state=42
)

X_train_under, y_train_under = rus.fit_resample(
    X_train,
    y_train
)

print(y_train_under.value_counts())

dissatisfied
0    9711
1    9711
Name: count, dtype: int64


## Model 1 - Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

lr_under = LogisticRegression(
    random_state=42,
    max_iter=5000
)

lr_under.fit(
    X_train_under,
    y_train_under
)

y_pred = lr_under.predict(X_test)

y_prob = lr_under.predict_proba(X_test)[:,1]

print("Logistic Regression - Undersampling")

print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1       :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

Logistic Regression - Undersampling
Accuracy : 0.8287544333280398
Precision: 0.380302580836547
Recall   : 0.5280065897858319
F1       : 0.4421451974478358
ROC AUC  : 0.7458152128065921


## Model 2 - Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_under = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

rf_under.fit(
    X_train_under,
    y_train_under
)

y_pred = rf_under.predict(X_test)

y_prob = rf_under.predict_proba(X_test)[:,1]

print("Random Forest - Undersampling")

print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1       :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

Random Forest - Undersampling
Accuracy : 0.791752686464454
Precision: 0.3246390312063344
Recall   : 0.5741350906095551
F1       : 0.4147575126450461
ROC AUC  : 0.7468855201334609


## Model 3 - XGBoost

In [9]:
from xgboost import XGBClassifier

xgb_under = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_under.fit(
    X_train_under,
    y_train_under
)

y_pred = xgb_under.predict(X_test)

y_prob = xgb_under.predict_proba(X_test)[:,1]

print("XGBoost - Undersampling")

print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1       :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

XGBoost - Undersampling
Accuracy : 0.7539039754380393
Precision: 0.2819556253681524
Recall   : 0.5914332784184514
F1       : 0.3818641138146523
ROC AUC  : 0.7362748111410731


## Threshold Tuning

In [10]:
models_to_tune = {
    'LR + Undersampling':  (lr_under, X_test),
    'RF + Undersampling':  (rf_under, X_test),
    'XGB + Undersampling': (xgb_under, X_test)
}

for name, (model, X) in models_to_tune.items():
    print(f"\n{name}")
    print(f"{'Threshold':<10} {'Recall':<10} {'Precision':<10} {'F1':<10}")

    y_prob_t = model.predict_proba(X)[:, 1]

    for threshold in [0.50, 0.45, 0.40, 0.35, 0.30]:
        y_pred_t = (y_prob_t >= threshold).astype(int)
        r = recall_score(y_test, y_pred_t)
        p = precision_score(y_test, y_pred_t)
        f = f1_score(y_test, y_pred_t)
        print(f"{threshold:<10} {r:.3f}     {p:.3f}     {f:.3f}")


LR + Undersampling
Threshold  Recall     Precision  F1        
0.5        0.528     0.380     0.442
0.45       0.593     0.323     0.418
0.4        0.673     0.245     0.359
0.35       0.802     0.179     0.293
0.3        0.937     0.139     0.243

RF + Undersampling
Threshold  Recall     Precision  F1        
0.5        0.587     0.315     0.410
0.45       0.648     0.259     0.370
0.4        0.723     0.209     0.324
0.35       0.808     0.175     0.288
0.3        0.889     0.153     0.260

XGB + Undersampling
Threshold  Recall     Precision  F1        
0.5        0.591     0.282     0.382
0.45       0.637     0.247     0.356
0.4        0.694     0.218     0.332
0.35       0.754     0.192     0.306
0.3        0.818     0.170     0.282


In [11]:
#Random Forest achieved the highest recall after threshold tuning.

#At a threshold of 0.35:

#Recall = 80.8%

#This makes Random Forest a strong candidate for deployment
#in a customer dissatisfaction prediction system where
#missing dissatisfied customers is costly.

#PHASE 8 - ARTIFICIAL NEURAL NETWORK (ANN)

In [12]:
#An Artificial Neural Network (ANN) was developed to evaluate
#whether deep learning could outperform traditional machine
#learning models for customer dissatisfaction prediction.

In [13]:
print(tf.__version__)

2.20.0


##Create SMOTE Data

In [14]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(
    X_train,
    y_train
)

##Build ANN

In [15]:
ann = Sequential([

    Dense(
        32,
        activation='relu',
        input_shape=(X_train_sm.shape[1],)
    ),

    Dense(
        16,
        activation='relu'
    ),

    Dense(
        8,
        activation='relu'
    ),

    Dense(
        1,
        activation='sigmoid'
    )

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##Compile Model

In [16]:
ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

##Train ANN

In [17]:
history = ann.fit(
    X_train_sm,
    y_train_sm,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6775 - loss: 0.8256 - val_accuracy: 0.4103 - val_loss: 0.8550
Epoch 2/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7256 - loss: 0.5619 - val_accuracy: 0.4939 - val_loss: 0.7380
Epoch 3/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7286 - loss: 0.5568 - val_accuracy: 0.5871 - val_loss: 0.6264
Epoch 4/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7293 - loss: 0.5574 - val_accuracy: 0.5219 - val_loss: 0.7173
Epoch 5/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7304 - loss: 0.5538 - val_accuracy: 0.4210 - val_loss: 0.9238
Epoch 6/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7337 - loss: 0.5483 - val_accuracy: 0.5257 - val_loss: 0.7445
Epoch 7/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7315 - loss: 0.5485 - val_accuracy: 0.5436 - val_loss: 0.7243
Epoch 8/20
412/412 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7348 - loss: 0.5437 - val_accuracy: 0.

##Evaluate ANN

In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_prob = ann.predict(X_test)

y_pred = (
    y_prob >= 0.5
).astype(int)

print("ANN Results")
print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1       :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

591/591 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
ANN Results
Accuracy : 0.8498756021385845
Precision: 0.4189189189189189
Recall   : 0.4341021416803954
F1       : 0.42637540453074435
ROC AUC  : 0.735050896919166


##ANN Threshold Tuning

In [19]:
print(f"{'Threshold':<10} {'Recall':<10} {'Precision':<10} {'F1':<10}")

for threshold in [0.50,0.45,0.40,0.35,0.30]:

    y_pred_t = (
        y_prob >= threshold
    ).astype(int)

    r = recall_score(y_test,y_pred_t)
    p = precision_score(y_test,y_pred_t)
    f = f1_score(y_test,y_pred_t)

    print(
        f"{threshold:<10}"
        f"{r:.3f}     "
        f"{p:.3f}     "
        f"{f:.3f}"
    )

Threshold  Recall     Precision  F1        
0.5       0.434     0.419     0.426
0.45      0.476     0.375     0.419
0.4       0.527     0.343     0.415
0.35      0.577     0.310     0.403
0.3       0.619     0.272     0.378


In [20]:
#Although ANN produced strong precision and F1 scores,
#it did not outperform Random Forest in recall.

#Therefore, Random Forest remained the preferred model for deployment.

#Save

In [21]:
import joblib

joblib.dump(
    rf_under,
    "/content/drive/MyDrive/Customer-Satisfaction-Retail-Analytics/models/final_rf_under.pkl"
)

print("Final model saved")

Final model saved
